# Notebook 04 - Soft-penalty neural survival model

A network that outputs cumulative hazard directly,

$$\Lambda(x, t) \ge 0, \qquad S(x, t) = e^{-\Lambda(x, t)},
  \qquad \lambda(x, t) = \frac{\partial \Lambda}{\partial t}$$

with a **penalty** in the loss that makes $\lambda < 0$ expensive.

## What this arm does and does not guarantee

**Guaranteed, by architecture.** The output layer is a Softplus, so
$\Lambda \ge 0$ everywhere, and therefore $S = e^{-\Lambda} \in (0, 1]$. That
is a genuine structural property: it holds for every input, at every point in
time, for any parameter values, before any training happens at all.

**Not guaranteed.** Monotonicity. The constraint $\lambda \ge 0$ is imposed by
adding $\alpha \cdot \mathrm{mean}\left[\mathrm{relu}(-\partial\Lambda/\partial t)^2\right]$
to the loss at sampled collocation points, with $\alpha = 0.1$. A penalty term
makes violations *costly*; it does not make them *impossible*. It is evaluated
at finitely many sampled points, it competes against the data likelihood, and it
is satisfied only to whatever degree the optimiser finds worthwhile. Between
collocation points, and anywhere the likelihood pulls hard enough, the network
can and does still emit a decreasing cumulative hazard.

An earlier version of this notebook described these curves as "guaranteed
monotonically decreasing" and the summary table recorded monotonicity as
"enforced". Neither claim is true of a soft penalty. The monotonicity audit at
the end of this notebook measures what the penalty actually achieves, and
whatever that number turns out to be is the result.

## On the earlier "physics-informed" framing

This arm was previously presented as a physics-informed neural network. The
relation being imposed, $dS/dt = -\lambda(t) S(t)$, is the *definition* of the
hazard function rearranged, not a governing physical law discovered about the
system. Nothing is being informed by physics: a definitional identity is being
used as a regulariser. The framing is dropped. The arm is a neural survival
model with a soft monotonicity penalty, which is what it has always been.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.common import (load_data, make_split, build_features, describe_split,
                        survival_arrays, subsample_train, HORIZONS, SEED,
                        SUBSAMPLE_N, RESULTS_DIR)
from src.evaluate import (evaluate_arm, save_arm_results, plot_calibration,
                          load_all_results, load_all_monotonicity)
from src.monotonicity import (audit_monotonicity, format_audit, plot_worst_curves,
                              audit_table)
from src.torch_arms import (set_seed, as_tensors, train_minibatch, plot_history,
                            batched_survival, batched_cumhaz)

set_seed(SEED)
RESULTS_DIR.mkdir(exist_ok=True)
print("torch", torch.__version__, "| threads", torch.get_num_threads())

## The shared protocol

Same split, same feature matrix as every other arm.

This also fixes a leak. The previous version of this notebook standardised the
full design matrix in cell 5 and split it in cell 7, so the scaler's means and
variances were computed over the test rows too. `src.common.build_features`
fits the scaler on train only.

**Training subsample.** As in notebook 03, this arm is fitted on a stratified
draw of `SUBSAMPLE_N = 300,000` training rows, because full-data training does
not finish in acceptable wall-clock here. The event rate is preserved, the size
is recorded in the results JSON, and the test split is untouched at 674,272
rows so every arm is scored on identical held-out data.

In [ ]:
df = load_data()
train_df_full, test_df = make_split(seed=SEED)
train_df = subsample_train(train_df_full, n=SUBSAMPLE_N)

print(f"full train rows      : {len(train_df_full):,}")
print(f"subsampled train rows: {len(train_df):,}  "
      f"(event rate {train_df['event'].mean():.6f} vs "
      f"{train_df_full['event'].mean():.6f} full)")
print(f"test rows            : {len(test_df):,}  (not subsampled)")
print()

feat = build_features(train_df, test_df)

t_tr, e_tr = survival_arrays(train_df)
t_te, e_te = survival_arrays(test_df)

print(describe_split(train_df, test_df).to_string(index=False))
print()
print("design matrix:", feat.X_train.shape, feat.X_test.shape)

X_tr, T_tr, E_tr = as_tensors(feat.X_train, t_tr, e_tr)
X_te, T_te, E_te = as_tensors(feat.X_test, t_te, e_te)

## Model - cumulative hazard network

Tanh hidden activations, because the loss differentiates the output with respect
to `t` and a ReLU network has a discontinuous derivative. Softplus output, which
is what makes $\Lambda \ge 0$ and hence $S \in (0, 1]$ hold structurally.

Time enters divided by 60, matching notebook 03. Autograd is taken with respect
to the `t` argument as passed in, so the chain rule returns
$\partial\Lambda/\partial t$ in per-month units regardless of the internal
rescaling.

In [ ]:
TIME_SCALE = 60.0

class CumulativeHazardNN(nn.Module):
    """
    Lambda(x, t) >= 0 via Softplus; S(x, t) = exp(-Lambda).

    Softplus guarantees Lambda >= 0, so S is a valid probability in (0, 1].
    It does NOT guarantee Lambda is non-decreasing in t; that is what the
    penalty term in the loss is for, and it is a soft constraint.
    """

    def __init__(self, d, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d + 1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1), nn.Softplus(),
        )

    def forward(self, x, t):
        """Cumulative hazard Lambda(x, t)."""
        return self.net(torch.cat([x, t / TIME_SCALE], dim=1))

    def survival(self, x, t):
        return torch.exp(-self.forward(x, t))

    def hazard(self, x, t):
        """lambda(x, t) = dLambda/dt by autograd, in per-month units."""
        t = t.clone().detach().requires_grad_(True)
        Lam = self.forward(x, t)
        return torch.autograd.grad(Lam.sum(), t, create_graph=True)[0]


model = CumulativeHazardNN(X_tr.shape[1])
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

## Loss

### Data likelihood

The right-censored survival negative log-likelihood is

$$-\ell = -\frac{1}{N}\sum_i \Big[\delta_i \log \lambda(x_i, t_i) - \Lambda(x_i, t_i)\Big]$$

The previous version computed the hazard as $\hat\lambda = \Lambda / t$. That
identity holds only when the hazard is constant in time, i.e. for an exponential
survival distribution. The whole point of this network is to learn a
time-varying cumulative hazard, so scoring it with a likelihood that assumes the
hazard is constant contradicts the model being fitted. The correct hazard was
already available: `physics_loss` computed $\partial\Lambda/\partial t$ by
autograd three lines away.

The hazard is now that autograd derivative. Inside the log it is clamped at
$10^{-8}$ so that a transiently negative $\lambda$ cannot produce a NaN; in that
region the gradient reaches the parameters through the $\Lambda$ term and
through the penalty below, which is what pushes $\lambda$ back up.

### Monotonicity penalty

$$\mathcal{L}_{\text{pen}} = \frac{1}{M}\sum_j
   \Big[\mathrm{relu}\big(-\partial\Lambda/\partial t \big|_{t_j}\big)\Big]^2,
\qquad \mathcal{L} = -\ell + \alpha\,\mathcal{L}_{\text{pen}}, \quad \alpha = 0.1$$

Collocation points are **resampled at every gradient step**: fresh borrowers
drawn from the minibatch, fresh times drawn uniformly from 0.1 to 60 months. The
previous version reused one fixed `torch.linspace(0.1, 60, 200)` grid for every
epoch, which lets the network satisfy the penalty on exactly those 200 abscissae
and do as it likes between them. Resampling removes that particular escape
route. It does not make the constraint hard.

In [ ]:
ALPHA = 0.1
T_MIN, T_MAX = 0.1, 60.0
N_COLLOC_BORROWERS, N_COLLOC_TIMES = 256, 32


def data_loss(model, x, t, e, eps=1e-8):
    """Right-censored survival NLL with the autograd hazard."""
    t_req = t.clone().detach().requires_grad_(True)
    Lam = model(x, t_req)
    lam = torch.autograd.grad(Lam.sum(), t_req, create_graph=True)[0]
    nll = -e * torch.log(lam.clamp_min(eps)) + Lam
    return nll.mean()


def penalty_loss(model, x_batch):
    """Soft monotonicity penalty on freshly sampled collocation points."""
    n = x_batch.shape[0]
    idx = torch.randint(0, n, (min(N_COLLOC_BORROWERS, n),))
    xc = x_batch[idx].detach().repeat_interleave(N_COLLOC_TIMES, dim=0)
    tc = (T_MIN + (T_MAX - T_MIN) * torch.rand(xc.shape[0], 1)).requires_grad_(True)
    Lam = model(xc, tc)
    grad = torch.autograd.grad(Lam.sum(), tc, create_graph=True)[0]
    return torch.relu(-grad).pow(2).mean()


def step_loss(model, xb, tb, eb):
    ld = data_loss(model, xb, tb, eb)
    lp = penalty_loss(model, xb)
    total = ld + ALPHA * lp
    return total, {"data": float(ld.detach()), "penalty": float(lp.detach())}

## Training

The same routine as notebook 03: minibatches of 8192, 10% of train held out,
patience 5, best weights restored. Early stopping watches the validation **data**
likelihood, since the penalty is a regulariser rather than a measure of fit.

The epoch ceiling is 60 and is **not** the intended stopping rule. What should
end training is early stopping, on a validation slice held out of train, once
the loss stops falling by a margin that matters (`min_delta=1e-4`). An arm that
runs into the ceiling instead has been cut off mid-descent, and reporting its
metrics would measure a truncated optimisation rather than the model. The run
prints which of the two happened, and `stopped_early` is recorded in the results
JSON so the comparison table can be checked.

In [ ]:
set_seed(SEED)
model = CumulativeHazardNN(X_tr.shape[1])

history = train_minibatch(model, step_loss, X_tr, T_tr, E_tr,
                          batch_size=8192, max_epochs=60, lr=1e-3,
                          val_frac=0.1, patience=5, seed=SEED, monitor="data")

In [ ]:
hist_df = pd.DataFrame({k: v for k, v in history.items() if isinstance(v, list)})
hist_df.to_csv(RESULTS_DIR / "nb04_training_history.csv", index=False)
print(hist_df[["epoch", "train_total", "train_data", "train_penalty",
               "val_data", "val_penalty", "steps", "seconds"]]
      .round(6).to_string(index=False))
print()
print(f"gradient steps taken: {history['total_steps']:,}  "
      f"(the previous version of this notebook took 30)")
print(f"stopped early: {history['stopped_early']}  "
      f"(False means the epoch ceiling bound and the arm is undertrained)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
plot_history(history, title="Soft-penalty NN - loss", ax=axes[0])
axes[1].semilogy(hist_df["epoch"], hist_df["train_penalty"], "o-", ms=3, label="train")
axes[1].semilogy(hist_df["epoch"], hist_df["val_penalty"], "s-", ms=3, label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("penalty term (log scale)")
axes[1].set_title("Monotonicity penalty during training")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Survival and hazard curves

Left: $S(t) = e^{-\Lambda(t)}$ for five held-out borrowers. Right: the
corresponding $\lambda(t) = \partial\Lambda/\partial t$. Any part of the right
panel below zero is a monotonicity violation, and the red line marks it.

In [ ]:
predict_survival = batched_survival(lambda x, t: model.survival(x, t))
predict_cumhaz = batched_cumhaz(lambda x, t: model(x, t))

grid = np.linspace(1, 60, 300)
Sc = predict_survival(feat.X_test[:5], grid)

xg = torch.as_tensor(np.repeat(feat.X_test[:5], len(grid), axis=0))
tg = torch.as_tensor(np.tile(grid, 5).reshape(-1, 1).astype(np.float32))
lam = model.hazard(xg, tg).detach().numpy().reshape(5, len(grid))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for i in range(5):
    axes[0].plot(grid, Sc[i], lw=1.5, label=f"borrower {i + 1}")
    axes[1].plot(grid, lam[i], lw=1.5)
axes[0].set_xlabel("Time (months)"); axes[0].set_ylabel("S(t)")
axes[0].set_title("Soft-penalty NN - survival curves")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].axhline(0, color="crimson", lw=1.4)
axes[1].set_xlabel("Time (months)"); axes[1].set_ylabel(r"$\lambda(t)=d\Lambda/dt$")
axes[1].set_title("Hazard (negative = violation)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("minimum hazard over these 5 borrowers:", float(lam.min()))
print("fraction of these plotted points with lambda < 0:",
      round(float((lam < 0).mean()), 6))

## Monotonicity audit

The measurement the study exists to produce, run here exactly as it is run for
every other arm: 1,000 held-out borrowers, cumulative hazard on a half-month
grid from 1 to 60 months, every point where `dLambda/dt < 0` counted.

In [ ]:
mono, detail = audit_monotonicity(predict_cumhaz, feat.X_test,
                                  name="Soft-penalty NN", n_borrowers=1000,
                                  return_detail=True)
print(format_audit(pd.DataFrame([mono])).T.to_string(header=False))

In [ ]:
if mono["pct_points_violating"] > 0:
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    plot_worst_curves(detail, n_curves=5, name="Soft-penalty NN", ax=ax)
    plt.tight_layout(); plt.show()
else:
    print("No violations found on the audit grid; nothing to plot.")

## Evaluation

In [ ]:
result = evaluate_arm(predict_survival, feat.X_test, test_df, train_df,
                     name="Soft-penalty NN")
print(result)
print()
print("notes:", result.notes)
save_arm_results(result, mono, extra={"alpha": ALPHA,
                                      "best_epoch": history["best_epoch"],
                                      "total_steps": history["total_steps"],
                                      "stopped_early": history["stopped_early"],
                                      "epochs_run": history["epoch"][-1],
                                      "max_epochs": history["max_epochs"],
                                      "subsample_n": SUBSAMPLE_N,
                                      "n_train_used": int(len(train_df))})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
plot_calibration(result, ax=axes[0])
axes[1].plot(result.brier["month"], result.brier["brier"], lw=1.8)
axes[1].set_xlabel("Time (months)"); axes[1].set_ylabel("IPCW Brier score")
axes[1].set_title("Brier score over time - Soft-penalty NN"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Arms completed so far

Every notebook writes its metrics to `results/`. This pulls back whatever has
been run at this point in the sequence. It is an interim view, not the study's
headline table: Cox PH (notebook 05) and the monotone-architecture network
(notebook 06) are not fitted yet, and the authoritative
`results/master_comparison.csv` is assembled at the end of notebook 06.

In [ ]:
comp = load_all_results().sort_values("c_harrell", ascending=False)
cols = ["arm", "c_harrell", "c_uno", "auc_12m", "auc_24m", "auc_36m", "ibs_1_60m"]
comp = comp[[c for c in cols if c in comp.columns]]
print("DISCRIMINATION AND ACCURACY")
print(comp.round(4).to_string(index=False))
comp.round(6).to_csv(RESULTS_DIR / "comparison_metrics.csv", index=False)

In [ ]:
audit = load_all_monotonicity()
audit = format_audit(audit).sort_values("pct_borrowers_violating")
print("MONOTONICITY VIOLATIONS")
print(audit.to_string(index=False))
audit.to_csv(RESULTS_DIR / "comparison_monotonicity.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
order = comp["arm"].tolist()
viol = audit.set_index("arm").reindex(order)["pct_borrowers_violating"]

axes[0].barh(order, comp["c_harrell"], color="steelblue")
axes[0].set_xlabel("Harrell's C"); axes[0].set_title("Discrimination")
axes[0].set_xlim(0.5, max(0.75, float(comp["c_harrell"].max()) + 0.02))
axes[0].grid(alpha=0.3, axis="x")

axes[1].barh(order, viol.values, color="indianred")
axes[1].set_xlabel("% of borrowers with >=1 monotonicity violation")
axes[1].set_title("Structural validity"); axes[1].grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

## Summary

| Property | Unconstrained NN (NB03) | Soft-penalty NN (this notebook) |
|---|---|---|
| $S \in (0,1)$ | Guaranteed, by sigmoid | Guaranteed, by Softplus on $\Lambda$ |
| $\Lambda \ge 0$ | Not applicable | **Guaranteed**, by Softplus |
| $S$ non-increasing in $t$ | Not imposed | **Penalised, not guaranteed** - see the audit above |
| Censoring | Binary cross-entropy; censored rows pushed to $S=1$ | Right-censored survival NLL |
| Hazard | Implicit in $S$ | $\lambda = \partial\Lambda/\partial t$ by autograd |

The row that matters is the third. A penalty makes violations expensive, not
impossible, and the audit reports how expensive turned out to be enough.

**What the numbers do not yet isolate.** This arm differs from notebook 03 in
two ways at once: the penalty, and the switch from binary cross-entropy to a
proper survival likelihood. The difference between them therefore measures the
pair, not the constraint. The missing control is this same architecture and
likelihood with $\alpha = 0$, and the missing upper comparison is an
architecture where monotonicity holds by construction rather than by penalty --
a monotonic network in $t$, whose violation rate is zero for structural reasons
and whose discrimination is the real answer to what a valid PD term structure
costs. Both are for the next pass.